In [0]:
# Silver Layer - Job friendly version
# All imports and table loads at the top

from pyspark.sql.functions import (
    col, to_timestamp, count, when, 
    sum, round, avg, max, desc, 
    month, year
)

# Load all bronze tables fresh
orders = spark.table("bronze_orders")
customers = spark.table("bronze_customers")
order_items = spark.table("bronze_order_items")
payments = spark.table("bronze_payments")
products = spark.table("bronze_products")

print("All bronze tables loaded!")
print("Orders:", orders.count())
print("Customers:", customers.count())
print("Order items:", order_items.count())
print("Payments:", payments.count())
print("Products:", products.count())

All bronze tables loaded!
Orders: 99441
Customers: 99441
Order items: 112650
Payments: 103886
Products: 32951


In [0]:
# Clean all tables

silver_orders = orders \
    .dropna(subset=["order_id", "customer_id", "order_status"]) \
    .filter(col("order_status") == "delivered") \
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date")))

silver_customers = customers \
    .dropna(subset=["customer_id", "customer_city", "customer_state"]) \
    .dropDuplicates(["customer_id"])

silver_order_items = order_items \
    .dropna(subset=["order_id", "product_id", "price"]) \
    .filter(col("price") > 0) \
    .filter(col("freight_value") >= 0)

silver_payments = payments \
    .dropna(subset=["order_id", "payment_type", "payment_value"]) \
    .filter(col("payment_value") > 0) \
    .dropDuplicates(["order_id", "payment_sequential"])

silver_products = products \
    .dropna(subset=["product_id", "product_category_name"])

print("All tables cleaned successfully!")
print("Clean orders:", silver_orders.count())
print("Clean customers:", silver_customers.count())

All tables cleaned successfully!
Clean orders: 96478
Clean customers: 99441


In [0]:
# Join all tables together and save as Silver master

spark.sql("DROP TABLE IF EXISTS silver_orders_master")

silver_master = silver_orders \
    .join(silver_customers, "customer_id", "left") \
    .join(silver_order_items, "order_id", "left") \
    .join(silver_payments, "order_id", "left") \
    .join(silver_products, "product_id", "left")

silver_master.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders_master")

print("Silver master table saved!")
print("Total rows:", silver_master.count())
print("Total columns:", len(silver_master.columns))

Silver master table saved!
Total rows: 115034
Total columns: 30


In [0]:
# Verify silver table is correct

silver = spark.table("silver_orders_master")
print("Silver table verified!")
print("Total rows:", silver.count())
print("Total columns:", len(silver.columns))
display(silver.limit(5))

Silver table verified!
Total rows: 115034
Total columns: 30


product_id,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
87285b34884572647811a353c7ac498a,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06T11:07:15.000Z,29.99,8.72,2,voucher,1,18.59,utilidades_domesticas,40,268,4,500,19,8,13
cafd558df4c3c9d1c338ba6930ea9a62,f70a0aff17df5a6cdd9a7196128bd354,456dc10730fbdba34615447ea195d643,delivered,2017-08-10T11:58:33.000Z,2017-08-12T02:45:24.000Z,2017-08-17T15:35:07.000Z,2017-08-18T14:28:02.000Z,2017-08-23T00:00:00.000Z,1974875b4a1d2e2ee6d586e3ba4d7602,5634,sao paulo,SP,1,5dceca129747e92ff8ef7a997dc4f8ca,2017-08-17T02:45:24.000Z,279.0,34.19,1,boleto,1,313.19,bebes,45,1009,1,16450,44,70,32
6b64362e89896be7589621df54be089e,989225ba6d0ebd5873335f7e01de2ae7,816f8653d5361cbf94e58c33f2502a5c,delivered,2017-12-12T13:56:04.000Z,2017-12-14T13:54:13.000Z,2017-12-16T00:18:57.000Z,2018-01-03T18:03:36.000Z,2018-01-08T00:00:00.000Z,37363700139c1aef873bbcd916e57dfd,5778,sao paulo,SP,1,77530e9772f57a62c906e1c21538ab82,2017-12-20T13:54:13.000Z,49.0,14.1,1,credit_card,1,63.1,moveis_decoracao,57,2435,2,3000,69,11,11
7564c1759c04fc0a38f2aa84f7a370ee,b52cc4919de82b4d696a4380d10804a3,be8c14c16a4d47194ccdfe10f1fc5b1a,delivered,2018-06-13T13:47:39.000Z,2018-06-15T02:37:29.000Z,2018-06-15T14:22:00.000Z,2018-06-18T22:32:44.000Z,2018-06-26T00:00:00.000Z,c86a25b8f5f6c203bb3471553bdc3200,13157,cosmopolis,SP,1,6860153b69cc696d5dcfe1cdaaafcf62,2018-06-19T02:30:26.000Z,42.99,12.03,1,boleto,1,55.02,construcao_ferramentas_construcao,59,2432,3,1200,16,11,11
2b4609f8948be18874494203496bc318,741fd1b80f015f170a7f458cf86769b8,df5c9e02596851fcbfc98804e4108bbd,delivered,2017-12-02T14:28:03.000Z,2017-12-02T14:38:30.000Z,2017-12-04T21:54:27.000Z,2017-12-11T16:17:04.000Z,2018-01-04T00:00:00.000Z,377bdfb2197d3b83c9e3f9502deabcfb,39680,capelinha,MG,1,cc419e0650a3c5ba77189a1882b7556a,2017-12-11T14:38:30.000Z,89.99,16.39,1,credit_card,1,106.38,beleza_saude,59,492,3,250,22,10,18
